In [1]:
# 필요한 라이브러리
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup

import time

In [4]:
def get_saramin_data():
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service)
    wait = WebDriverWait(driver, 10)

    driver.get('https://www.saramin.co.kr/')

    search_btn = wait.until(EC.element_to_be_clickable((By.ID, 'btn_search')))
    search_btn.click()

    search_input = wait.until(EC.presence_of_element_located((By.ID, 'ipt_keyword_recruit')))
    search_input.send_keys("데이터분석")

    submit_btn = wait.until(EC.element_to_be_clickable((By.ID, 'btn_search_recruit')))
    submit_btn.click()

    time.sleep(2)

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    job_section = soup.select_one('section.section_search#recruit_info')
    job_list = job_section.select('div#recruit_info_list > div.content > div.item_recruit')

    result_list = []
    for job in job_list:
        try:
            site = "Saramin"
            company_tag = job.select_one('div.area_corp strong.corp_name a')
            company = company_tag.text.strip() if company_tag else None

            title_tag = job.select_one('h2.job_tit a')
            title = title_tag.get('title').strip() if title_tag else None

            keyword_tags = job.select('div.job_condition span')
            details = ' / '.join([span.get_text(strip=True) for span in keyword_tags]) if keyword_tags else None

            url_tail = title_tag.get('href') if title_tag else None
            full_url = f"https://www.saramin.co.kr{url_tail}" if url_tail else None

            result_list.append({
                'Site': site,
                'Col_Company': company,
                'Col_Recruit': title,
                'Col_detail': details,
                'Col_url': full_url
            })
        except:
            continue

    driver.quit()
    
    return pd.DataFrame(result_list)

In [6]:
saramin_df = get_saramin_data()
saramin_df.to_csv('data_saramin.csv', index=False, encoding='utf-8-sig')